# 从零实现教学版 3D Gaussian Splatting：投影、排序与可微合成

3D Gaussian Splatting 把场景表示为带位置、尺度、旋转、颜色和不透明度的高斯集合。渲染时把 3D covariance 经相机 Jacobian 投影成 2D 椭圆，按深度排序，再做 front-to-back alpha compositing。

本 Notebook 用纯 PyTorch 手写一个 CPU Tiny renderer 和受控优化。它明确省略正式系统的 CUDA tile rasterizer、球谐颜色、visibility-aware densification/pruning、抗锯齿和大规模相机标定；小图拟合结果不能冒充真实新视角合成质量。

In [ ]:
import copy
import hashlib
import json
import math
import random
import warnings
from types import MappingProxyType

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")
import torch
from torch import nn
import torch.nn.functional as F

SEED60=6001
random.seed(SEED60); torch.manual_seed(SEED60); torch.set_num_threads(1)
DEVICE60=torch.device("cpu")
def canonical_json60(value): return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(",",":"))
assert DEVICE60.type=="cpu" and torch.get_num_threads()==1
assert torch.__version__.split(".")[0]>="2"

## 1. 相机合同与解析投影

相机坐标定义为 $p_c=R_{cw}(p_w-t)$，相机朝 `+z` 看；只有 $z>near$ 可见。像素中心坐标为 $u=f_x x/z+c_x, v=f_y y/z+c_y$。`R_cw` 的行分别是相机 right/up/forward 轴，必须近似正交且行列式为 +1。viewport、焦距、主点和方向都必须进入发布制品，不能只存一张渲染图。

In [ ]:
class Camera60:
    def __init__(self,position,R_cw,fx,fy,cx,cy,height,width,near=.1):
        self.position=torch.as_tensor(position,dtype=torch.float32); self.R_cw=torch.as_tensor(R_cw,dtype=torch.float32)
        self.fx,self.fy,self.cx,self.cy=float(fx),float(fy),float(cx),float(cy); self.height,self.width=int(height),int(width); self.near=float(near)
        if self.position.shape!=(3,) or self.R_cw.shape!=(3,3) or self.fx<=0 or self.fy<=0 or min(self.height,self.width)<2 or self.near<=0: raise ValueError("invalid_camera")
        eye=self.R_cw@self.R_cw.T
        if not torch.allclose(eye,torch.eye(3),atol=1e-5) or not torch.allclose(torch.det(self.R_cw),torch.tensor(1.),atol=1e-5): raise ValueError("camera_rotation_not_SO3")
    def manifest(self):
        return {"position":self.position.tolist(),"R_cw":self.R_cw.tolist(),"fx":self.fx,"fy":self.fy,"cx":self.cx,"cy":self.cy,"height":self.height,"width":self.width,"near":self.near}

def project_points60(points,camera):
    if points.ndim!=2 or points.shape[1]!=3 or not torch.isfinite(points).all(): raise ValueError("point_contract")
    pc=(points-camera.position)@camera.R_cw.T; z=pc[:,2]; safe_z=torch.where(z>camera.near,z,torch.ones_like(z))
    uv=torch.stack([camera.fx*pc[:,0]/safe_z+camera.cx,camera.fy*pc[:,1]/safe_z+camera.cy],1)
    valid=z>camera.near  # 只按相机前方裁剪；屏外中心的椭圆尾部仍可能与 viewport 相交
    return uv,z,valid,pc

def center_in_view60(uv,camera):
    return (uv[:,0]>=-.5)&(uv[:,0]<camera.width-.5)&(uv[:,1]>=-.5)&(uv[:,1]<camera.height-.5)

camera60=Camera60([0,0,0],torch.eye(3),8,8,7.5,7.5,16,16)
uv60,z60,valid60,pc60=project_points60(torch.tensor([[0.,0.,2.],[.25,-.5,2.],[0.,0.,-1.]]),camera60)
assert torch.allclose(uv60[:2],torch.tensor([[7.5,7.5],[8.5,5.5]])) and torch.equal(valid60,torch.tensor([True,True,False]))
offscreen_uv60,_,offscreen_front60,_=project_points60(torch.tensor([[3.,0.,1.]]),camera60)
assert offscreen_front60.item() and not center_in_view60(offscreen_uv60,camera60).item()
away60=Camera60([0,0,0],torch.diag(torch.tensor([-1.,1.,-1.])),8,8,7.5,7.5,16,16)
assert not project_points60(torch.tensor([[0.,0.,2.]]),away60)[2].item()
assert camera60.manifest()["width"]==16 and camera60.near==.1
try:
    Camera60([0,0,0],torch.ones(3,3),8,8,7.5,7.5,16,16)
    raise AssertionError("non-SO3 camera should fail")
except ValueError as exc: assert "SO3" in str(exc)

## 2. 3D rotation/scale 与 Jacobian covariance 投影

3D covariance 为 $\Sigma_3=R_g\operatorname{diag}(s_x^2,s_y^2,s_z^2)R_g^T$。教学版只学习绕 z 轴角度；投到相机后用透视 Jacobian $J$ 得 $\Sigma_2=J R_{cw}\Sigma_3R_{cw}^T J^T+\epsilon I$。`softplus(log_scale)` 保证尺度为正，$\epsilon$ 防止求逆奇异，但它不是掩盖 NaN 的借口。

In [ ]:
def rotation_z60(angles):
    c,s=angles.cos(),angles.sin(); z=torch.zeros_like(c); o=torch.ones_like(c)
    return torch.stack([c,-s,z,s,c,z,z,z,o],-1).reshape(-1,3,3)

def covariance_3d60(scales,angles):
    if scales.ndim!=2 or scales.shape[1]!=3 or angles.shape!=(len(scales),) or (scales<=0).any(): raise ValueError("invalid_gaussian_scale")
    r=rotation_z60(angles); return r@torch.diag_embed(scales.square())@r.transpose(-2,-1)

def project_covariance60(pc,cov_world,camera,min_variance=1e-4):
    if pc.shape!=(len(cov_world),3) or (pc[:,2]<=camera.near).any(): raise ValueError("covariance_requires_visible_points")
    x,y,z=pc.unbind(1); zero=torch.zeros_like(z)
    j=torch.stack([camera.fx/z,zero,-camera.fx*x/z.square(),zero,camera.fy/z,-camera.fy*y/z.square()],1).reshape(-1,2,3)
    r=camera.R_cw.expand(len(pc),-1,-1); cov_cam=r@cov_world@r.transpose(-2,-1)
    cov2=j@cov_cam@j.transpose(-2,-1)+min_variance*torch.eye(2)[None]
    return cov2,j

scales_oracle60=torch.tensor([[.1,.2,.3],[.3,.1,.2]])
cov3_oracle60=covariance_3d60(scales_oracle60,torch.tensor([0.,math.pi/3]))
eig3_60=torch.linalg.eigvalsh(cov3_oracle60)
assert eig3_60.min()>0 and torch.allclose(torch.det(rotation_z60(torch.tensor([.7]))),torch.ones(1),atol=1e-6)
cov2_oracle60,j_oracle60=project_covariance60(torch.tensor([[0.,0.,2.]]),cov3_oracle60[:1],camera60)
assert torch.allclose(cov2_oracle60[0],torch.diag(torch.tensor([.1601,.6401])),atol=1e-4)
assert torch.linalg.eigvalsh(cov2_oracle60).min()>0 and j_oracle60.shape==(1,2,3)
assert torch.allclose(cov2_oracle60,cov2_oracle60.transpose(-2,-1),atol=1e-7)

tilt60=.35
R_tilt60=torch.tensor([[math.cos(tilt60),0.,math.sin(tilt60)],[0.,1.,0.],[-math.sin(tilt60),0.,math.cos(tilt60)]])
camera_tilt60=Camera60([.2,-.1,.3],R_tilt60,9,7,7.5,7.5,16,16)
pc_offaxis60=torch.tensor([[.35,-.25,2.2]])
world_offaxis60=pc_offaxis60@R_tilt60+camera_tilt60.position
uv_offaxis60,_,valid_offaxis60,pc_roundtrip60=project_points60(world_offaxis60,camera_tilt60)
anisotropic_cov60=covariance_3d60(torch.tensor([[.12,.25,.18]]),torch.tensor([.6]))
cov2_offaxis60,j_offaxis60=project_covariance60(pc_roundtrip60,anisotropic_cov60,camera_tilt60,min_variance=0.)
def project_camera_point60(pc):
    return torch.stack([camera_tilt60.fx*pc[...,0]/pc[...,2]+camera_tilt60.cx,camera_tilt60.fy*pc[...,1]/pc[...,2]+camera_tilt60.cy],-1)
eps60=1e-3; fd_columns60=[]
for axis60 in range(3):
    delta60=torch.zeros_like(pc_offaxis60); delta60[0,axis60]=eps60
    fd_columns60.append(((project_camera_point60(pc_offaxis60+delta60)-project_camera_point60(pc_offaxis60-delta60))/(2*eps60))[0])
j_fd60=torch.stack(fd_columns60,1)
cov_camera60=R_tilt60@anisotropic_cov60[0]@R_tilt60.T
cov2_fd60=j_fd60@cov_camera60@j_fd60.T
assert valid_offaxis60.item() and not torch.allclose(uv_offaxis60[0],torch.tensor([7.5,7.5]))
assert anisotropic_cov60[0,0,1].abs()>1e-4 and torch.allclose(j_offaxis60[0],j_fd60,atol=2e-3,rtol=2e-3)
assert torch.allclose(cov2_offaxis60[0],cov2_fd60,atol=3e-3,rtol=3e-3) and torch.linalg.eigvalsh(cov2_offaxis60).min()>0

class GaussianProjector60(nn.Module):
    def __init__(self,min_variance=1e-3): super().__init__(); self.min_variance=float(min_variance)
    def forward(self,means,scales,angles,camera):
        uv,depth,valid,pc=project_points60(means,camera); visible=torch.where(valid)[0]; order=visible[depth[visible].argsort()]
        if len(order):
            cov3=covariance_3d60(scales[order],angles[order]); cov2,_=project_covariance60(pc[order],cov3,camera,self.min_variance)
        else: cov2=means.new_empty((0,2,2))
        return uv,depth,valid,pc,order,cov2

projector_probe60=GaussianProjector60()
projected_probe60=projector_probe60(torch.tensor([[0.,0.,2.]]),torch.tensor([[.1,.2,.3]]),torch.zeros(1),camera60)
assert projected_probe60[4].tolist()==[0] and projected_probe60[5].shape==(1,2,2)

## 3. 椭圆权重与 front-to-back alpha compositing

像素与投影中心差为 $d$，高斯权重 $w=\exp(-\tfrac12d^T\Sigma_2^{-1}d)$，有效 alpha 为 `opacity*w`。由近到远累积：$C\leftarrow C+T\alpha c, T\leftarrow T(1-\alpha)$，最后加 $T C_{bg}$。普通加权平均不满足遮挡；排序反了会让远处物体覆盖近处物体。

In [ ]:
def composite_samples60(colors,alphas,depths,background):
    if colors.ndim!=2 or colors.shape[1]!=3 or alphas.shape!=depths.shape!=(len(colors),): raise ValueError("composite_shape")
    if (alphas<0).any() or (alphas>1).any() or not torch.isfinite(colors).all(): raise ValueError("invalid_composite_values")
    order=depths.argsort(); trans=torch.ones((),dtype=colors.dtype); out=torch.zeros(3,dtype=colors.dtype)
    for idx in order:
        out=out+trans*alphas[idx]*colors[idx]; trans=trans*(1-alphas[idx])
    return out+trans*background,1-trans,order

colors_oracle60=torch.tensor([[1.,0.,0.],[0.,0.,1.]])
near_red60,alpha_oracle60,order_oracle60=composite_samples60(colors_oracle60,torch.tensor([.8,.8]),torch.tensor([1.,2.]),torch.zeros(3))
near_blue60,_,_=composite_samples60(colors_oracle60,torch.tensor([.8,.8]),torch.tensor([2.,1.]),torch.zeros(3))
assert near_red60[0]>near_red60[2] and near_blue60[2]>near_blue60[0]
transparent60,transparent_alpha60,_=composite_samples60(colors_oracle60,torch.zeros(2),torch.tensor([1.,2.]),torch.tensor([.2,.3,.4]))
assert torch.allclose(transparent60,torch.tensor([.2,.3,.4])) and transparent_alpha60==0
assert torch.equal(order_oracle60,torch.tensor([0,1]))
assert 0<alpha_oracle60<1

## 4. 可微 Gaussian scene 参数化

`TinyGaussianScene60` 学习 mean、softplus scale、z-rotation、sigmoid RGB 与 sigmoid opacity。约束变换必须在 forward 中，而不是每步训练后原地截断参数，否则 optimizer 状态与参数语义会分离。这里 gaussian 数固定；正式 3DGS 会根据 view-space 梯度 densify、split、clone 和 prune。

In [ ]:
class TinyGaussianScene60(nn.Module):
    def __init__(self,count=3):
        super().__init__()
        if count<1: raise ValueError("scene_must_have_gaussians")
        self.count=count; self.means=nn.Parameter(torch.randn(count,3)*.1+torch.tensor([0.,0.,2.]))
        self.raw_scales=nn.Parameter(torch.full((count,3),-2.2)); self.angles=nn.Parameter(torch.zeros(count))
        self.color_logits=nn.Parameter(torch.zeros(count,3)); self.opacity_logits=nn.Parameter(torch.ones(count))
    def forward(self):
        scales=F.softplus(self.raw_scales)+.025; colors=self.color_logits.sigmoid(); opacity=self.opacity_logits.sigmoid()*.98
        return self.means,scales,self.angles,colors,opacity

scene_probe60=TinyGaussianScene60(4); means_p60,scales_p60,angles_p60,colors_p60,opacity_p60=scene_probe60()
assert means_p60.shape==(4,3) and scales_p60.min()>0
assert ((colors_p60>0)&(colors_p60<1)).all() and ((opacity_p60>0)&(opacity_p60<1)).all()

## 5. Tiny differentiable renderer

renderer 对可见 gaussian 按中心深度排序，再在完整小 viewport 上评估每个椭圆。复杂度是 $O(GHW)$，仅适合教学；正式实现把屏幕分 tile、裁剪到有限 support，并用 CUDA 并行。背景参与最后一步合成，返回 `(rgb, accumulated_alpha, diagnostics)`。

In [ ]:
class TinyGaussianRenderer60(nn.Module):
    def __init__(self,min_variance=1e-3): super().__init__(); self.min_variance=float(min_variance); self.projector=GaussianProjector60(min_variance)
    def forward(self,scene,camera,background):
        means,scales,angles,colors,opacity=scene()
        if background.shape!=(3,) or not torch.isfinite(background).all(): raise ValueError("background_contract")
        uv,depth,valid,pc,order,cov2=self.projector(means,scales,angles,camera)
        yy,xx=torch.meshgrid(torch.arange(camera.height,dtype=means.dtype),torch.arange(camera.width,dtype=means.dtype),indexing="ij")
        grid=torch.stack([xx,yy],-1); rgb=torch.zeros(camera.height,camera.width,3,dtype=means.dtype); trans=torch.ones(camera.height,camera.width,dtype=means.dtype)
        if len(order):
            diff=grid[None]-uv[order,None,None,:]; inv=torch.linalg.inv(cov2)
            mahal=torch.einsum("ghwi,gij,ghwj->ghw",diff,inv,diff)
            alpha=(opacity[order,None,None]*torch.exp(-.5*mahal)).clamp(0,.99)
            one_minus=1-alpha
            trans_before=torch.cat([torch.ones_like(one_minus[:1]),torch.cumprod(one_minus[:-1],dim=0)],dim=0)
            rgb=(trans_before[...,None]*alpha[...,None]*colors[order,None,None,:]).sum(0)
            trans=one_minus.prod(0)
        rgb=rgb+trans[...,None]*background
        if not torch.isfinite(rgb).all() or not torch.isfinite(trans).all(): raise FloatingPointError("nonfinite_render")
        return rgb,1-trans,{"order":order,"uv":uv,"depth":depth,"valid":valid,"center_in_view":center_in_view60(uv,camera)}

renderer60=TinyGaussianRenderer60(); bg60=torch.tensor([.02,.02,.02])
rendered_probe60,alpha_probe60,diag_probe60=renderer60(scene_probe60,camera60,bg60)
assert rendered_probe60.shape==(16,16,3) and alpha_probe60.shape==(16,16)
assert torch.isfinite(rendered_probe60).all() and ((alpha_probe60>=0)&(alpha_probe60<=1)).all()
assert torch.equal(diag_probe60["depth"][diag_probe60["order"]].sort().values,diag_probe60["depth"][diag_probe60["order"]])
assert diag_probe60["valid"].dtype==torch.bool and diag_probe60["uv"].shape==(4,2)
assert diag_probe60["center_in_view"].shape==(4,)

## 6. 梯度、遮挡、背景与 viewport oracle

除解析投影/PSD/排序测试外，还必须验证 render 对位置、尺度、旋转、颜色和 opacity 可微。相机背后的 gaussian 被排除；屏外中心不能直接等价于无贡献，本教学 renderer 会继续评估其椭圆尾部。全透明场景应严格返回背景。注意近裁剪与 depth sort 在边界处不可微，这是 rasterizer 的固有分段结构，而不是 autograd bug。

In [ ]:
gradient_scene60=TinyGaussianScene60(2)
with torch.no_grad():
    gradient_scene60.means.copy_(torch.tensor([[-.25,-.15,2.0],[.35,.2,2.4]]))
    gradient_scene60.raw_scales.copy_(torch.tensor([[-1.2,-2.2,-2.7],[-2.0,-1.1,-2.5]]))
    gradient_scene60.angles.copy_(torch.tensor([.4,-.55]))
grad_rgb60,grad_alpha60,grad_diag60=renderer60(gradient_scene60,camera60,bg60)
gy60,gx60=torch.meshgrid(torch.linspace(-1,1,16),torch.linspace(-1,1,16),indexing="ij")
((grad_rgb60[...,0]*(gx60+0.3*gy60)).sum()+grad_alpha60.mean()).backward()
for parameter60 in [gradient_scene60.means,gradient_scene60.raw_scales,gradient_scene60.angles,gradient_scene60.color_logits,gradient_scene60.opacity_logits]:
    assert parameter60.grad is not None and torch.isfinite(parameter60.grad).all() and parameter60.grad.abs().sum()>0
assert gradient_scene60.angles.grad.abs().min()>1e-7
behind_scene60=TinyGaussianScene60(1)
with torch.no_grad(): behind_scene60.means.copy_(torch.tensor([[0.,0.,-2.]])); behind_scene60.opacity_logits.fill_(-50)
behind_rgb60,behind_alpha60,behind_diag60=renderer60(behind_scene60,camera60,bg60)
assert torch.allclose(behind_rgb60,bg60.expand_as(behind_rgb60)) and behind_alpha60.max()==0
assert not behind_diag60["valid"].item()

## 7. 受控 inverse rendering

先用三个已知 gaussian 渲染 target，再从偏移的位置、灰色颜色和较低 opacity 出发优化同样数量的 gaussian。单相机小图存在多解，loss 下降只验证可微链路；它不能证明恢复了真实 3D geometry，更不能代表 novel-view quality。正式实验必须保留独立相机 split。

In [ ]:
target_scene60=TinyGaussianScene60(3)
with torch.no_grad():
    target_scene60.means.copy_(torch.tensor([[-.45,-.2,2.0],[.35,.15,2.25],[.05,.48,2.6]]))
    target_scene60.raw_scales.copy_(torch.tensor([[-1.65,-1.8,-2.2],[-1.8,-1.55,-2.1],[-1.75,-1.75,-2.0]]))
    target_scene60.angles.copy_(torch.tensor([.25,-.5,.8]))
    target_scene60.color_logits.copy_(torch.tensor([[3.,-2.,-2.],[-2.,3.,-2.],[-2.,-2.,3.]])); target_scene60.opacity_logits.fill_(2.2)
    target_image60=renderer60(target_scene60,camera60,bg60)[0].detach()

torch.manual_seed(SEED60+2); model60=TinyGaussianScene60(3)
with torch.no_grad():
    model60.means.copy_(target_scene60.means+torch.tensor([[.12,-.08,0.],[.08,.1,0.],[-.1,.06,0.]]))
optimizer60=torch.optim.Adam(model60.parameters(),lr=.035); losses60=[]
for step60 in range(41):
    optimizer60.zero_grad(); predicted60,_,_=renderer60(model60,camera60,bg60)
    loss60=F.mse_loss(predicted60,target_image60); loss60.backward(); optimizer60.step(); losses60.append(float(loss60.detach()))
final_image60,final_alpha60,final_diag60=renderer60(model60,camera60,bg60)
assert losses60[-1]<losses60[0]*.18 and losses60[-1]<5e-4
assert torch.isfinite(final_image60).all() and final_diag60["valid"].all()
assert len(losses60)==41 and torch.isfinite(final_alpha60).all()
print({"initial":round(losses60[0],6),"final":round(losses60[-1],6),"note":"single-view controlled fit, not generalization"})

## 8. 绑定 camera/data/recipe 的发布制品

state 摘要绑定每个参数的 key/dtype/shape/bytes；manifest 绑定 target image、固定相机、背景、split、线性 RGB/像素中心约定与优化 recipe。loader 对照包外只读 publisher registry，返回只允许批准 camera 的 wrapper。即使攻击者替换状态、相机并重算包内整体哈希，也无法改变 registry 的发布摘要。

In [ ]:
def tensor_digest60(t):
    t=t.detach().cpu().contiguous(); h=hashlib.sha256(); h.update(str(t.dtype).encode()); h.update(canonical_json60(list(t.shape)).encode()); h.update(t.numpy().tobytes()); return h.hexdigest()
def state_digest60(state):
    h=hashlib.sha256()
    for key in sorted(state):
        t=state[key].detach().cpu().contiguous(); h.update(key.encode()); h.update(str(t.dtype).encode()); h.update(canonical_json60(list(t.shape)).encode()); h.update(t.numpy().tobytes())
    return h.hexdigest()
def artifact_digest60(package):
    signed={"subject":package["subject"],"manifest":package["manifest"],"state_digest":state_digest60(package["state"])}
    return hashlib.sha256(canonical_json60(signed).encode()).hexdigest()

manifest60={
    "config":{"count":3,"renderer_min_variance":1e-3},"camera":camera60.manifest(),
    "data":{"target_image":tensor_digest60(target_image60),"background":tensor_digest60(bg60)},
    "split":{"train_camera_ids":["camera-identity-16"],"validation_camera_ids":[],"warning":"single-camera controlled oracle"},
    "preprocess":{"color":"linear-RGB","pixel_centers":"integer coordinates","world_to_camera":"R_cw@(p-t)","forward":"+z"},
    "recipe":{"seed":SEED60+2,"optimizer":"Adam","steps":41,"lr":.035,"loss":"pixel-MSE","topology":"fixed-3-gaussians"},
}
package60={"subject":"tiny-gaussian-scene60/v1","manifest":copy.deepcopy(manifest60),"state":copy.deepcopy(model60.state_dict())}
package60["artifact_digest"]=artifact_digest60(package60)
_PUBLISHER_REGISTRY60=MappingProxyType({package60["subject"]:package60["artifact_digest"]})

def deep_freeze60(value):
    if isinstance(value,dict): return MappingProxyType({key:deep_freeze60(item) for key,item in value.items()})
    if isinstance(value,(list,tuple)): return tuple(deep_freeze60(item) for item in value)
    return value

class PublishedRenderer60:
    def __init__(self,scene,renderer,manifest):
        self._scene=scene.eval(); self._renderer=renderer.eval()
        self._camera_json=canonical_json60(manifest["camera"]); self._background_digest=manifest["data"]["background"]
        self.manifest=deep_freeze60(copy.deepcopy(manifest))
    @torch.no_grad()
    def render(self,camera,background):
        if canonical_json60(camera.manifest())!=self._camera_json: raise ValueError("camera_not_published")
        if tensor_digest60(background)!=self._background_digest: raise ValueError("background_not_published")
        return self._renderer(self._scene,camera,background)

def load_published60(package):
    actual=artifact_digest60(package)
    if _PUBLISHER_REGISTRY60.get(package.get("subject"))!=actual: raise ValueError("publisher_digest_mismatch")
    if package.get("artifact_digest")!=actual or package["manifest"]!=manifest60: raise ValueError("manifest_mismatch")
    scene=TinyGaussianScene60(package["manifest"]["config"]["count"]); scene.load_state_dict(package["state"],strict=True)
    renderer=TinyGaussianRenderer60(package["manifest"]["config"]["renderer_min_variance"])
    return PublishedRenderer60(scene,renderer,package["manifest"])

published60=load_published60(package60); pub_render60=published60.render(camera60,bg60)[0]
assert torch.allclose(pub_render60,final_image60,atol=1e-7) and isinstance(published60.manifest,MappingProxyType)
published_fx60=published60.manifest["camera"]["fx"]
try:
    published60.manifest["camera"]["fx"]=999.
    raise AssertionError("nested camera manifest was mutable")
except TypeError: pass
try:
    published60.manifest["recipe"]["lr"]=9.
    raise AssertionError("nested recipe was mutable")
except TypeError: pass
assert published60.manifest["camera"]["fx"]==published_fx60==8.0
forged60=copy.deepcopy(package60); forged60["manifest"]["camera"]["fx"]=999.; forged60["state"]["means"].add_(1)
forged60["artifact_digest"]=artifact_digest60(forged60)
try:
    load_published60(forged60); raise AssertionError("overall re-sign should fail")
except ValueError as exc: assert "publisher" in str(exc)

## 9. 复杂度、失败模式与生产差距

- 本实现以张量计算完整 viewport，复杂度为 $O(GHW)$；它只按 near plane 剔除，屏外中心的 Gaussian 尾部仍会正确贡献，但也会浪费计算。正式 rasterizer 应用椭圆 bounding box 与 viewport 求交后再做 tile binning、有限 support、排序和 GPU kernel。显存还受可见 Gaussian 数及反向缓存支配。
- 常见错误：把 `R_wc` 当 `R_cw`、相机朝向符号颠倒、Jacobian 漏掉 `-fx*x/z²` 项、covariance 非 PSD、按远到近合成、忽略透明背景、像素中心偏半格。
- 硬排序、视锥边界和 topology change 不连续；需配合 densification/pruning 策略、opacity reset 与稳定数值实现。单视角 loss 很低仍可能是 billboard 式歧义。
- 正式系统还需要多相机标定版本、曝光/颜色管理、SH、LOD、抗锯齿、资源上限、真实签名和回滚。这里没有实现 tile CUDA、SH 或 PyTorch3D/nerfstudio 依赖。

## 10. 原始资料

- Kerbl et al., [3D Gaussian Splatting for Real-Time Radiance Field Rendering](https://arxiv.org/abs/2308.04079)
- Zwicker et al., [EWA Splatting](https://vcg.seas.harvard.edu/publications/ewa-splatting)
- PyTorch 官方文档：[Autograd mechanics](https://pytorch.org/docs/stable/notes/autograd.html)

原论文的实时结果依赖专用 rasterizer 与完整训练策略；本 Notebook 只复现可验证的几何与合成核心。